# 🔍 Notebook 5 — RAG Engine (Q&A on Contracts)
## What this notebook does
Builds the Q&A system. You ask a question, it finds the right contract clauses and answers with citations.

**Simple explanation:**  
Instead of reading 150 pages to find the payment clause,  
you just type "what is the payment deadline?" and get  
the exact answer with the document and section number.

**Technical explanation:**  
Retrieval Augmented Generation (RAG) pipeline:  
1. STORE: Convert all contract chunks to vector embeddings using  
   sentence-transformers (all-MiniLM-L6-v2). Store in ChromaDB.  
2. RETRIEVE: Convert user query to embedding. Find top-5 most  
   semantically similar chunks using cosine similarity.  
3. GENERATE: Pass retrieved chunks as context to Groq LLM.  
   LLM generates answer grounded in actual contract content.  
   Answer includes document name and clause reference citation.


## Step 1 — Setup

In [ ]:
import os, time
from groq import Groq
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
import chromadb
from sqlalchemy import create_engine, text as sql_text
from sqlalchemy.orm import sessionmaker

load_dotenv()
client  = Groq(api_key=os.getenv("GROQ_API_KEY"))
engine  = create_engine("sqlite:///database/contractiq.db", echo=False)
Session = sessionmaker(bind=engine)
session = Session()

# Load the embedding model
# Simple:    A small AI that converts text to numbers
# Technical: Sentence-BERT variant. 384-dimensional embeddings.
#            80MB download. Runs on CPU. ~0.1s per chunk.
print("Loading embedding model (first time = download ~80MB)...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("✅ Embedding model loaded")

# Setup ChromaDB
# Simple:    A special database that stores text with meaning,
#            so you can search by similarity not just keywords
# Technical: Persistent vector store with cosine similarity index.
chroma_client     = chromadb.PersistentClient(path="./vectorstore")
collection_name   = "contract_chunks"

# Get or create collection
try:
    collection = chroma_client.get_collection(collection_name)
    print(f"✅ ChromaDB collection loaded: {collection.count()} chunks already stored")
except:
    collection = chroma_client.create_collection(
        name      = collection_name,
        metadata  = {"hnsw:space": "cosine"}  # cosine similarity
    )
    print("✅ ChromaDB collection created (empty)")

## Step 2 — Store Embeddings

**Simple:** Convert all contract text chunks into numbers and save them in ChromaDB.

**Technical:** Batch encode chunks using SentenceTransformer. Store document vectors, text, and metadata (doc_name, chunk_index) in ChromaDB persistent collection. Skip chunks already stored.

In [ ]:
def store_embeddings():
    """
    Converts all contract chunks to vectors and stores in ChromaDB.
    
    Simple:    Takes every paragraph from every contract and
               converts it to a list of numbers that represents
               its meaning. Stores in ChromaDB for fast search.
    
    Technical: SentenceTransformer.encode() produces 384-dim
               float32 vectors. Batch size 32 for memory efficiency.
               ChromaDB stores vectors with IDs for later retrieval.
    """
    # Load chunks from SQLite
    chunks = session.execute(sql_text(
        "SELECT id, doc_name, chunk_text, chunk_index FROM chunks"
    )).fetchall()
    
    if not chunks:
        print("❌ No chunks found. Run Notebook 2 first.")
        return
    
    # Check what's already in ChromaDB
    existing_count = collection.count()
    print(f"📊 Chunks in SQLite   : {len(chunks)}")
    print(f"📊 Chunks in ChromaDB : {existing_count}")
    
    if existing_count >= len(chunks):
        print("✅ All chunks already embedded. Skipping.")
        return
    
    # Get existing IDs to avoid duplicates
    if existing_count > 0:
        existing = collection.get()
        existing_ids = set(existing["ids"])
    else:
        existing_ids = set()
    
    # Filter to new chunks only
    new_chunks = [c for c in chunks if f"chunk_{c[0]}" not in existing_ids]
    print(f"\n⚙️  Embedding {len(new_chunks)} new chunks...")
    
    batch_size = 32
    for i in range(0, len(new_chunks), batch_size):
        batch = new_chunks[i:i+batch_size]
        
        texts      = [c[2] for c in batch]
        ids        = [f"chunk_{c[0]}" for c in batch]
        metadatas  = [{"doc_name": c[1], "chunk_index": c[3]} for c in batch]
        
        # Generate embeddings
        embeddings = embed_model.encode(texts, show_progress_bar=False)
        
        # Store in ChromaDB
        collection.add(
            documents  = texts,
            embeddings = embeddings.tolist(),
            ids        = ids,
            metadatas  = metadatas,
        )
        
        print(f"   Batch {i//batch_size + 1}: {len(batch)} chunks embedded ✅")
    
    print(f"\n✅ Total chunks in ChromaDB: {collection.count()}")

store_embeddings()

## Step 3 — The Retrieval Function

**Simple:** Given a question, finds the 5 most relevant contract paragraphs.

**Technical:** Encodes query with same SentenceTransformer model. Computes cosine similarity against all stored vectors. Returns top-k results with distance scores and metadata.

In [ ]:
def retrieve_relevant_chunks(query, top_k=5):
    """
    Finds the most relevant contract chunks for a given question.
    
    Simple:    Like a very smart search engine that understands
               meaning, not just keywords. 'payment deadline' finds
               'invoice must be settled within 45 days' even though
               the words are completely different.
    
    Technical: Encodes query to 384-dim vector. ChromaDB computes
               cosine similarity against all stored vectors.
               Returns top_k chunks ranked by similarity score.
               Includes metadata for citation (doc_name, chunk_index).
    """
    if collection.count() == 0:
        return []
    
    # Convert question to vector
    query_embedding = embed_model.encode([query]).tolist()
    
    # Search ChromaDB
    results = collection.query(
        query_embeddings = query_embedding,
        n_results        = min(top_k, collection.count()),
        include          = ["documents", "metadatas", "distances"]
    )
    
    chunks = []
    for i in range(len(results["documents"][0])):
        similarity = 1 - results["distances"][0][i]   # convert distance to similarity
        chunks.append({
            "text"        : results["documents"][0][i],
            "doc_name"    : results["metadatas"][0][i]["doc_name"],
            "chunk_index" : results["metadatas"][0][i]["chunk_index"],
            "similarity"  : round(similarity, 3),
        })
    
    return chunks

# Test retrieval
print("🔍 Testing retrieval with sample question:\n")
test_query  = "What is the uptime commitment?"
test_results = retrieve_relevant_chunks(test_query, top_k=3)

for i, r in enumerate(test_results, 1):
    print(f"  Result {i} | Similarity: {r['similarity']} | Doc: {r['doc_name']}")
    print(f"  Text: {r['text'][:200]}...")
    print()

## Step 4 — The Answer Generator

**Simple:** Takes the relevant chunks and asks Groq to write a clear answer with sources.

**Technical:** Constructs few-shot RAG prompt. Context window = top-5 chunks (~2000 tokens). Instructs LLM to answer ONLY from context (prevents hallucination). Forces citation of document and clause.

In [ ]:
RAG_SYSTEM = """You are a helpful contract assistant. You answer questions about contracts
based ONLY on the provided contract excerpts. 
Always cite which document and clause your answer comes from.
If the answer is not in the provided excerpts, say 'I could not find this in the provided contracts.'
Never make up information."""

def answer_question(question):
    """
    Full RAG pipeline: retrieve → generate → cite.
    
    Simple:    1. Find the relevant contract paragraphs
               2. Give them to the AI as context
               3. Ask the AI to answer based only on those paragraphs
               4. Return answer + which document it came from
    
    Technical: Zero-shot RAG with citation enforcement.
               Context = top-5 chunks concatenated with metadata.
               Temperature = 0.2 for factual consistency.
               Returns dict with answer, sources, and confidence.
    """
    print(f"🔍 Retrieving relevant chunks for: '{question}'")
    
    # Step 1: Retrieve
    chunks = retrieve_relevant_chunks(question, top_k=5)
    
    if not chunks:
        return {
            "answer"  : "No contract documents have been loaded yet.",
            "sources" : [],
        }
    
    # Step 2: Build context
    context_parts = []
    sources       = []
    
    for i, chunk in enumerate(chunks, 1):
        context_parts.append(
            f"[SOURCE {i}]\n"
            f"Document: {chunk['doc_name']}\n"
            f"Relevance: {chunk['similarity']}\n"
            f"Text: {chunk['text']}\n"
        )
        sources.append({
            "doc"        : chunk["doc_name"],
            "similarity" : chunk["similarity"],
            "preview"    : chunk["text"][:100]
        })
    
    context = "\n---\n".join(context_parts)
    
    # Step 3: Generate answer
    prompt = f"""Based ONLY on the contract excerpts below, answer this question:

QUESTION: {question}

CONTRACT EXCERPTS:
{context}

INSTRUCTIONS:
- Answer clearly and directly
- Always mention which document (SOURCE number) the answer comes from
- If multiple documents have relevant info, mention all of them
- If documents contradict each other, flag it as a conflict
- Keep answer under 200 words

ANSWER:"""
    
    response = client.chat.completions.create(
        model      = "llama3-70b-8192",
        messages   = [
            {"role": "system", "content": RAG_SYSTEM},
            {"role": "user",   "content": prompt}
        ],
        max_tokens  = 400,
        temperature = 0.2,
    )
    
    answer = response.choices[0].message.content.strip()
    
    return {
        "question" : question,
        "answer"   : answer,
        "sources"  : sources,
    }

print("✅ answer_question() defined")

## Step 5 — Test the Q&A System

Try asking questions about your contracts!

In [ ]:
# Test questions — these should all be answerable from your contracts
test_questions = [
    "What is the payment deadline?",
    "What is the uptime commitment?",
    "What happens in case of a data breach?",
    "What is the liability cap?",
    "When does the contract renew?",
]

for question in test_questions:
    print(f"{'='*60}")
    result = answer_question(question)
    print(f"\n❓ QUESTION: {result['question']}")
    print(f"\n💬 ANSWER:\n{result['answer']}")
    print(f"\n📎 SOURCES:")
    for s in result['sources'][:2]:
        print(f"   • {s['doc']} (similarity: {s['similarity']})")
    print()
    time.sleep(1)  # brief pause between calls

print("\n✅ RAG Engine working!")
print("\n▶ Now run: streamlit run demo_app.py")

## Step 6 — Interactive Q&A (type your own questions)

In [ ]:
# Run this cell and type your own question
question = input("Ask anything about your contracts: ")

if question.strip():
    result = answer_question(question)
    print(f"\n💬 ANSWER:\n{result['answer']}")
    print(f"\n📎 SOURCES:")
    for s in result["sources"][:3]:
        print(f"   • {s['doc']} (relevance: {s['similarity']})")
        print(f"     Preview: {s['preview']}...")
else:
    print("Please type a question!")